# Experiment Tracking Notebook
**Course:** AL2002 — Artificial Intelligence  
**Project:** Intelligent Reading Comprehension and Quiz Generation System  

This notebook tracks all experiments run during the project, including hyperparameter variations, feature ablation studies, and model comparisons.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
import joblib
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB

sys.path.append('../')
print('Libraries loaded successfully.')

In [ ]:
# Load processed data
X_train = sparse.load_npz('../data/processed/train_ohe.npz')
y_train = np.load('../data/processed/train_labels.npy', allow_pickle=True)
X_val = sparse.load_npz('../data/processed/val_ohe.npz')
y_val = np.load('../data/processed/val_labels.npy', allow_pickle=True)

print(f'Train: {X_train.shape}, Val: {X_val.shape}')

## Experiment 1: Baseline Results (OHE, max_features=5000)

In [ ]:
# Experiment 1 — Final results on validation set from model_a_train.py
experiment_results = pd.DataFrame([
    {'Model': 'Logistic Regression', 'Accuracy': 0.2291, 'Macro_F1': 0.2246, 'Train_Time_s': 42.5},
    {'Model': 'SVM (LinearSVC)',     'Accuracy': 0.2237, 'Macro_F1': 0.2226, 'Train_Time_s': 52.8},
    {'Model': 'Naive Bayes',         'Accuracy': 0.2341, 'Macro_F1': 0.2323, 'Train_Time_s': 1.1},
    {'Model': 'Ensemble (Voting)',   'Accuracy': 0.2249, 'Macro_F1': 0.2228, 'Train_Time_s': 73.2},
    {'Model': 'Random Baseline',     'Accuracy': 0.25,   'Macro_F1': 0.25,   'Train_Time_s': 0},
])

print(experiment_results.to_string(index=False))

In [ ]:
# Visualization — Model Comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

models = experiment_results['Model']
x = range(len(models))

axes[0].bar(x, experiment_results['Accuracy'], color=['#4c8bf5','#34a853','#fbbc05','#ea4335','#aaa'])
axes[0].axhline(0.25, color='red', linestyle='--', label='Random Baseline (25%)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=15, ha='right')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model A — Accuracy Comparison (Validation Set)')
axes[0].legend()

axes[1].bar(x, experiment_results['Macro_F1'], color=['#4c8bf5','#34a853','#fbbc05','#ea4335','#aaa'])
axes[1].axhline(0.25, color='red', linestyle='--', label='Random Baseline (25%)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=15, ha='right')
axes[1].set_ylabel('Macro F1')
axes[1].set_title('Model A — Macro F1 Comparison (Validation Set)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../report/model_a_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to report/model_a_comparison.png')

## Experiment 2: K-Means Clustering Results

In [ ]:
# K-Means was trained with n_clusters=4 (matching A/B/C/D labels)
kmeans_results = pd.DataFrame([
    {'k': 4, 'Purity': 0.2736, 'Silhouette': -0.034, 'Note': 'Final (matches class count)'}
])
print(kmeans_results.to_string(index=False))
print('\nA negative Silhouette Score confirms OHE vectors form overlapping clusters.')
print('This is expected for bag-of-words representations on a diverse dataset.')

## Conclusions from Experiments

- All Traditional ML models with OHE features perform near the 25% random baseline
- Naive Bayes is the best performer (23.41% accuracy) while being 40x faster to train
- K-Means clustering produces overlapping clusters (negative Silhouette), confirming that OHE features do not capture semantic groupings
- These results motivate the use of deep learning for reading comprehension tasks